# Measuring a credit model against the maximum achievable score

A credit model scores **0.84** on a lender's book. Is that good?

Nobody knows. The data has a ceiling too — some borrowers default for reasons
nothing in the file predicts — and that ceiling is invisible. So every argument
about model quality on real data is an argument about an unknown denominator.

This notebook removes the unknown. We generate a portfolio whose risk driver we
**chose**, keep it hidden from the model, and compute two numbers no real dataset
can offer:

| | |
|---|---|
| **Ceiling** | the best score obtainable from the columns a model can see |
| **Oracle** | what a model that could see the hidden driver would score |

Then `0.84` becomes *"0.84 against a ceiling of 0.87"* — a statement about the
**model** instead of a number floating free.

Runs in about a minute on a laptop. No GPU, no API keys, no data agreement.

## 1. What is being generated, and what is hidden

Everything below comes from one file: **`packs/credit_benchmark_known_ceiling.yaml`**.
Nothing is hardcoded in this notebook. Read that file alongside this section and the
whole experiment is legible.

### The causal chain

```
  risk_tier          A B C D E, hidden from the model
      │
      ├─────────────────────────────┐
      ▼                             ▼
  three noisy readings          a stress multiplier
  bureau_score                  A 0.35  …  E 4.50
  revolving_utilisation_pct         │
  debt_to_income_pct                ▼
      │                       scales the monthly
      │                       arrears transitions
      │                             │
      │                             ▼
      │                       arrears_state
      │                    Performing → 1-29 → 30-59 → 60-89 → Defaulted
      │                             │
      ▼                             ▼
  what the model sees        the outcome it must predict
```

A borrower's tier decides two things at once: **how they look** (the three readings)
and **what happens to them** (how fast they fall behind). The model sees the first
and is asked to predict the second. That is the entire experiment.

### The three column roles that matter

Every column in the YAML carries a `role`, and three of them decide what a model may
see:

| role | in the output file? | what it is here |
|---|---|---|
| `helper` | **no — dropped before writing** | `risk_tier`, and the three `_noise_*` draws |
| `derived` | yes | the three observables, computed from tier + noise |
| `static` / `dynamic` | yes | `loan_id`, `region`, `current_balance`, `arrears_state` |

`helper` is the mechanism that makes this work. Those columns are generated, used to
drive everything, and then **deleted before the file is written** — so they cannot
leak into a model even by accident.

#### The hidden columns, in full

| column | what it holds | why hidden |
|---|---|---|
| `risk_tier` | A–E, drawn 30/28/22/13/7% | the answer. A model that saw it would score ~0.92 and prove nothing |
| `_noise_score` | Gaussian, σ = 46 | the measurement error on `bureau_score` |
| `_noise_util` | Gaussian, σ = 13 | the measurement error on `revolving_utilisation_pct` |
| `_noise_dti` | Gaussian, σ = 7.5 | the measurement error on `debt_to_income_pct` |
| `_centre_score`, `_centre_util`, `_centre_dti` | the per-tier means | intermediate values in the emission model |

Those σ values are the **difficulty dial**. Section 5 turns it.

### How an observable is built

Each observable is *its tier's centre plus independent noise*. In the YAML:

```yaml
derivations:
  - target: _centre_score          # the mean for each tier
    kind: when
    rules:
      - {if: "risk_tier == 'A'", then: 786}
      - {if: "risk_tier == 'B'", then: 731}
      - {if: "risk_tier == 'C'", then: 676}
      - {if: "risk_tier == 'D'", then: 621}
    else: 566                      # tier E

  - target: bureau_score           # what the model actually sees
    kind: expr
    expr: _centre_score + _noise_score
```

So a tier-C borrower has a bureau score centred on 676, scattered by σ = 46. A
tier-B borrower centres on 731. **The distributions overlap heavily**, which is why a
single reading is not enough and why the ceiling is well below 1.0.

Utilisation and DTI work identically but point the other way — higher means worse.

**Their errors are independent**, and that is deliberate: it is what makes combining
all three worth more than reading the best one, and therefore what puts the ceiling
above any single feature.

### Two columns that are deliberately useless

```yaml
- name: region            # North / South / East / West
  description: A decoy. Independent of the hidden tier and of the outcome.

- name: current_balance   # drawn independently of everything
  description: Outstanding principal. Independent of the hidden tier.
```

Neither carries any signal. They are there so you can check whether a model, or a
feature-attribution report, claims they do. **Anything that appears to gain from them
has found noise** — and on a real portfolio you would never be able to tell.

### How the tier reaches the outcome

This is the part that is easiest to miss in the YAML, so it is worth stating plainly.

The arrears ladder has one baseline transition matrix — what a mid-tier borrower does
each month. The tier does **not** get its own matrix. Instead it carries a multiplier
that scales the *worsening* half of that matrix, per borrower:

```yaml
secondary_chains:
  - name: tier
    lifecycle:
      state_column: risk_tier
      transitions: [identity]    # the tier never changes
    coupling:
      stress: {A: 0.35, B: 0.70, C: 1.40, D: 2.60, E: 4.50}
```

A tier-A borrower slips at about a third of the baseline rate; a tier-E borrower at
four and a half times it. The chain uses an **identity matrix** — a borrower keeps the
tier they were born with — and exists purely to carry that multiplier.

Compounded over 18 months, those multipliers produce the default rates you will see in
section 3: roughly **0.25% for tier A and 77% for tier E**. Nothing declares those
numbers; they fall out of the matrix and the multipliers together, which is why the
notebook measures them rather than asserting them.

### One honest limitation of this pack

The three observables are set **at origination and never revised** — in YAML terms
their derivations run at `stage: book`, period 0 only.

That makes this a *cross-sectional* problem wearing a panel's clothes: all the
information is in row one, and the remaining 17 cut-offs add none. It is why logistic
regression attains the ceiling in section 4 — with a single snapshot and Gaussian
readings, the optimal rule genuinely is linear.

**So this pack cannot demonstrate the value of a sequence model.** A benchmark where
the readings deteriorate month by month — utilisation creeping up before default —
would separate a model that reads history from one that reads a row. That is a
different pack, and a worthwhile one.

In [1]:
import pandas as pd

from sdd import api, benchmark

PACK = "credit_benchmark_known_ceiling"
spec = api.load(PACK)

result = api.run(PACK, 20_000, "./out/benchmark", seed=7, validate_output=False)
panel = pd.read_parquet(result["panel"])

print(
    f"{result['total_rows']:,} rows · {result['entities']:,} loans · {result['periods']} cut-offs"
)
print("columns a model may see:", list(panel.columns))

359,999 rows · 20,000 loans · 18 cut-offs
columns a model may see: ['loan_id', 'as_of_date', 'bureau_score', 'revolving_utilisation_pct', 'debt_to_income_pct', 'region', 'current_balance', 'arrears_state']


Note what is **not** in that list. `risk_tier` was generated, drove every outcome,
and was dropped before the file was written. That asymmetry — you know the answer,
the model does not — is the entire instrument.

Worth demonstrating rather than taking on trust:

In [2]:
declared = {c.name: c.role for c in spec.columns}
hidden = [name for name, role in declared.items() if role == "helper"]

print("declared as hidden in the YAML:")
for name in hidden:
    print(f"   {name:16} role={declared[name]:8} in the panel? {name in panel.columns}")

print()
print("the benchmark block names the driver and how to invert it:")
print(f"   latent      {spec.benchmark.latent}")
for o in spec.benchmark.observables:
    print(f"   observable  {o.column:26} centres={o.centres:15} noise={o.noise}")
print(f"   outcome     reaching any of {spec.benchmark.label_states}")

declared as hidden in the YAML:
   risk_tier        role=helper   in the panel? False
   _noise_score     role=helper   in the panel? False
   _noise_util      role=helper   in the panel? False
   _noise_dti       role=helper   in the panel? False

the benchmark block names the driver and how to invert it:
   latent      risk_tier
   observable  bureau_score               centres=_centre_score   noise=_noise_score
   observable  revolving_utilisation_pct  centres=_centre_util    noise=_noise_util
   observable  debt_to_income_pct         centres=_centre_dti     noise=_noise_dti
   outcome     reaching any of ['60-89 DPD', 'Defaulted']


Every hidden column reports `False` — generated, used, gone.

The `benchmark` block is what turns the pack from a dataset into an instrument. It
names the driver, points at the centres and noise widths for each observable, and
states what counts as a bad outcome. That is exactly the information needed to invert
the generating process, which is what section 3 does.

### What the data actually looks like

One row per loan per cut-off. Here is a single borrower's 18 months, and then the
opening cut-off across borrowers — which is all a model gets to see.

In [3]:
one = panel[panel["loan_id"] == panel["loan_id"].iloc[0]]
print(f"one borrower, {len(one)} monthly rows:")
display(
    one[
        [
            "as_of_date",
            "arrears_state",
            "bureau_score",
            "revolving_utilisation_pct",
            "debt_to_income_pct",
            "current_balance",
        ]
    ].head(6)
)

print("the three readings are fixed at origination — see the limitation note above")
print("arrears_state is what moves, and it is the outcome, not a feature")

one borrower, 18 monthly rows:


,as_of_date,arrears_state,bureau_score,revolving_utilisation_pct,debt_to_income_pct,current_balance
0,2024-01-31,Performing,627.0,21.25,31.79,18187.21
20000,2024-02-29,Performing,627.0,21.25,31.79,18187.21
40000,2024-03-31,Performing,627.0,21.25,31.79,18187.21
60000,2024-04-30,Performing,627.0,21.25,31.79,18187.21
80000,2024-05-31,Performing,627.0,21.25,31.79,18187.21
100000,2024-06-30,Performing,627.0,21.25,31.79,18187.21


the three readings are fixed at origination — see the limitation note above
arrears_state is what moves, and it is the outcome, not a feature


## 2. The question a model is asked

*Given only what is visible at the first cut-off, which borrowers go on to reach
60+ days past due at any point?*

Features are read at the opening cut-off only. Reading later cut-offs would leak
the answer — the arrears state **is** the outcome.

In [4]:
X = benchmark.observables(spec, panel)  # opening cut-off, declared observables only
y = benchmark.label_outcome(spec, panel)  # ever reached 60-89 DPD or Defaulted

print(f"{len(X):,} borrowers · bad rate {y.mean():.2%}")
X.head()

20,000 borrowers · bad rate 12.91%


,bureau_score,revolving_utilisation_pct,debt_to_income_pct
loan_id,,,
L00000001,627.0,21.25,31.79
L00000002,643.0,79.85,19.48
L00000003,667.0,39.73,25.52
L00000004,761.0,33.59,25.02
L00000005,761.0,28.43,34.35


## 3. The ceiling

This is the part real data cannot give you.

Because the generating process is declared, the posterior over the hidden tier
given the three readings is exact — Bayes' rule on independent Gaussians. The best
possible score is the posterior mean of the outcome rate. It is *derived*, not
estimated, which is why it settles arguments instead of starting them.

In [5]:
from sklearn.model_selection import train_test_split

# Out-of-sample, always. See section 6 for what happens otherwise.
train_ids, test_ids = train_test_split(X.index, test_size=0.4, random_state=0, stratify=y)
held_out = panel[panel["loan_id"].isin(set(test_ids))]

known = benchmark.ceiling(spec, held_out)
print(known.summary())
print()
print("P(bad | hidden tier) — measured by generating once with the tier exposed:")
for tier, risk in known.latent_risk.items():
    print(f"   {tier}   {risk:6.2%}")

ceiling 0.8987 · oracle 0.9162 (observables cost 0.0175) · outcome rate 12.912%

P(bad | hidden tier) — measured by generating once with the tier exposed:
   A    0.25%
   B    1.27%
   C    9.20%
   D   37.32%
   E   77.44%


Two numbers, and the distance between them matters:

- the **oracle** is what perfect knowledge of the driver would buy;
- the **ceiling** is what the noisy readings actually support.

The gap is information the observables do not carry. **No model closes it** — it is
a property of the data, not of anyone's algorithm. Knowing its size tells you
whether to buy a better model or better data, which is usually the more expensive
question to get wrong.

## 4. Three models against it

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = {
    "bureau score alone": None,
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "gradient boosting": HistGradientBoostingClassifier(max_iter=400, random_state=0),
}

rows = []
for name, model in candidates.items():
    if model is None:
        scores = -X.loc[test_ids, "bureau_score"]  # lower score, higher risk
    else:
        model.fit(X.loc[train_ids], y.loc[train_ids])
        scores = pd.Series(model.predict_proba(X.loc[test_ids])[:, 1], index=test_ids)
    rows.append(benchmark.compare(spec, held_out, scores, name=name))

report = pd.DataFrame(rows)[
    ["name", "achieved", "ceiling", "captured", "gap_to_ceiling", "beat_the_ceiling"]
]
report.style.format(
    {"achieved": "{:.4f}", "ceiling": "{:.4f}", "captured": "{:.1%}", "gap_to_ceiling": "{:+.4f}"}
)

,name,achieved,ceiling,captured,gap_to_ceiling,beat_the_ceiling
0,bureau score alone,0.8656,0.8987,91.7%,+0.0331,False
1,logistic regression,0.8987,0.8987,100.0%,-0.0000,False
2,gradient boosting,0.8923,0.8987,98.4%,+0.0064,False


### Reading the result

**`captured`** is the number worth quoting. It is the share of *available* signal
the model found, where available means "above a coin flip, up to the ceiling".
A raw AUC cannot be compared across datasets; this can.

Three things usually show up here, and all three are hard to see without a ceiling:

1. **The single feature leaves signal on the table.** It reads the best one of the
   three and ignores the rest.
2. **Logistic regression lands on the ceiling.** Not luck — with equal-variance
   Gaussian readings the optimal boundary genuinely *is* linear, so the simplest
   model is the right one. Without a ceiling you would never learn that and would
   keep buying complexity.
3. **Gradient boosting scores slightly lower.** More capacity, fitting noise. On
   real data this looks like a tuning problem you could fix; here you can see it
   is a problem with no solution, because the simpler model is already optimal.

## 5. Turning the difficulty dial

The noise widths in the pack decide how much the observables reveal. Widen them and
the ceiling falls toward chance; narrow them and it climbs toward the oracle.

That is what makes this a *measuring instrument* rather than a fixed dataset: you
can manufacture a problem of **known difficulty** and check that an evaluation
pipeline reports that difficulty correctly. A validation process that cannot tell
an easy problem from a hard one will not tell you much about a real one either.

In [7]:
import copy

dial = []
for factor in (0.5, 1.0, 2.0):
    variant = copy.deepcopy(spec.model_dump(mode="json", exclude_none=True, by_alias=True))
    for column in variant["columns"]:
        if column["name"].startswith("_noise_"):
            column["generator"]["stddev"] *= factor
    noisy = pd.read_parquet(
        api.run(variant, 20_000, f"./out/dial_{factor}", seed=11, validate_output=False)["panel"]
    )
    measured = benchmark.ceiling(variant, noisy)
    dial.append(
        {
            "noise": f"{factor:.1f}x",
            "ceiling": measured.ceiling,
            "oracle": measured.oracle,
            "observables_cost": measured.observable_cost,
        }
    )

pd.DataFrame(dial).style.format(
    {"ceiling": "{:.4f}", "oracle": "{:.4f}", "observables_cost": "{:.4f}"}
)

,noise,ceiling,oracle,observables_cost
0,0.5x,0.9158,0.9162,0.0004
1,1.0x,0.9015,0.9162,0.0147
2,2.0x,0.8476,0.9162,0.0686


The oracle barely moves — the hidden driver is unchanged, so perfect knowledge is
worth the same. The **ceiling** moves a lot. Halve the measurement error and more
of the truth becomes recoverable; double it and it does not.

That is the honest version of "our model got worse on the new data".

> **On the low-noise row.** The ceiling can edge a hair above the oracle there, and
> that is not a contradiction — the two are measured on different samples, and when
> the readings are precise enough to pin the tier down they converge to the same
> quantity. A crossing of a few thousandths is sampling error meeting a gap that has
> genuinely closed. It is worth showing rather than smoothing: these are
> measurements, and measurements have error bars.

## 6. The check that keeps this honest

A ceiling is a claim, and a claim that cannot fail is worth nothing. So
`compare()` reports `beat_the_ceiling` — because a model **cannot** legitimately
beat the best achievable score. If one does, either the ceiling is wrong or the
model saw something it should not have.

It caught a real mistake while this notebook was being written: scoring the
gradient booster on the same rows it was trained on gave **0.93** against a
ceiling of **0.897**. That is not a strong model, it is a memorised one — and on
real data it would have looked like a triumph.

Here it is, reproduced deliberately.

In [8]:
leaky = HistGradientBoostingClassifier(max_iter=400, random_state=0).fit(X, y)
in_sample = pd.Series(leaky.predict_proba(X)[:, 1], index=X.index)

caught = benchmark.compare(spec, panel, in_sample, name="scored in-sample")
print(f"achieved {caught['achieved']:.4f}  vs ceiling {caught['ceiling']:.4f}")
print("flagged as impossible:", caught["beat_the_ceiling"])

achieved 0.9301  vs ceiling 0.8974
flagged as impossible: True


On a real portfolio that result is indistinguishable from success. Here it is
arithmetically impossible, and the instrument says so.

## What this is for

**Measuring a model.** Not "is 0.84 good?" but "0.84 of an available 0.87 —
the model found 97% of the signal that exists."

**Rehearsing a validation process.** Run your whole evaluation pipeline —
leakage checks, out-of-time splits, challenger comparison, sign-off — on a problem
whose answer is known, before any real data is involved. A process that cannot
recover a known answer is not ready for an unknown one.

**Comparing across datasets.** Raw AUC is not comparable between portfolios
because the ceilings differ. Share-of-available-signal is.

**Before the data conversation.** All of the above runs on a laptop, on day one,
with nothing to sign.

---

### What it does not tell you

Worth being blunt, because the limit is real: **a good score here does not mean a
model will work on a real book.** This data was made by rules we wrote, and a model
that excels at recovering them has recovered *our rules*.

What it measures is whether a model extracts the signal that is present, and
whether an evaluation process is sound. Both transfer. Predicted performance does
not — for that, point the same pipeline at your own tape, which is one YAML file
away.